# Check CAMERA_HIGH Count Against Video Frames

Point this notebook at a completed run directory. It will:
- count `CAMERA_HIGH` rows in `serial.csv`
- count frames in the selected video file with OpenCV
- report the difference and a few extra serial sanity markers

In [5]:
from __future__ import annotations

import csv
from pathlib import Path

import cv2

# Point this at a completed run directory.
RUN_DIR = Path("/home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52/")

# Pick the video file you want to compare against serial.csv.
# Examples: 'raw.mp4', 'raw_cam1.mp4'
VIDEO_NAME = "raw.mp4"

In [6]:
def read_serial_rows(run_dir: Path) -> list[dict[str, str]]:
    path = run_dir / "serial.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing serial.csv: {path}")
    with path.open(newline="") as fh:
        return list(csv.DictReader(fh))


def count_video_frames(video_path: Path) -> int:
    if not video_path.exists():
        raise FileNotFoundError(f"Missing video file: {video_path}")
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")
    try:
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    finally:
        cap.release()
    return frame_count


def rows_by_event(rows: list[dict[str, str]], event_name: str) -> list[dict[str, str]]:
    return [row for row in rows if (row.get("eventType") or "").strip() == event_name]


def last_count(rows: list[dict[str, str]]) -> int | None:
    if not rows:
        return None
    raw = (rows[-1].get("count") or "").strip()
    if not raw:
        return None
    try:
        return int(raw)
    except ValueError:
        return None


In [7]:
serial_rows = read_serial_rows(RUN_DIR)
video_path = RUN_DIR / VIDEO_NAME
video_frame_count = count_video_frames(video_path)

camera_high_rows = rows_by_event(serial_rows, "CAMERA_HIGH")
camera_low_rows = rows_by_event(serial_rows, "CAMERA_LOW")
ack_start_rows = rows_by_event(serial_rows, "ACK_START")
ack_stop_rows = rows_by_event(serial_rows, "ACK_STOP")

camera_high_count = len(camera_high_rows)
camera_low_count = len(camera_low_rows)
ack_stop_frame_count = last_count(ack_stop_rows)

summary = {
    "run_dir": str(RUN_DIR),
    "video": VIDEO_NAME,
    "video_frame_count": video_frame_count,
    "camera_high_rows": camera_high_count,
    "camera_low_rows": camera_low_count,
    "camera_high_minus_video": camera_high_count - video_frame_count,
    "ack_start_rows": len(ack_start_rows),
    "ack_stop_rows": len(ack_stop_rows),
    "ack_stop_count_field": ack_stop_frame_count,
}

summary

{'run_dir': '/home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52',
 'video': 'raw.mp4',
 'video_frame_count': 242,
 'camera_high_rows': 242,
 'camera_low_rows': 242,
 'camera_high_minus_video': 0,
 'ack_start_rows': 1,
 'ack_stop_rows': 1,
 'ack_stop_count_field': 242}

In [8]:
print(f"Run: {RUN_DIR}")
print(f"Video: {VIDEO_NAME}")
print(f"Video frame count: {video_frame_count}")
print(f"CAMERA_HIGH rows: {camera_high_count}")
print(f"CAMERA_LOW rows: {camera_low_count}")
print(f"Difference (CAMERA_HIGH - video): {camera_high_count - video_frame_count}")
print(f"ACK_START rows: {len(ack_start_rows)}")
print(f"ACK_STOP rows: {len(ack_stop_rows)}")
print(f"ACK_STOP count field: {ack_stop_frame_count}")

if camera_high_rows:
    print("\nFirst CAMERA_HIGH row:")
    print(camera_high_rows[0])
    print("\nLast CAMERA_HIGH row:")
    print(camera_high_rows[-1])

if ack_stop_rows:
    print("\nLast ACK_STOP row:")
    print(ack_stop_rows[-1])

Run: /home/jetson/Desktop/SqueakView/runs/GoNoGo/rewt_2026-05-11_09-59-52
Video: raw.mp4
Video frame count: 242
CAMERA_HIGH rows: 242
CAMERA_LOW rows: 242
Difference (CAMERA_HIGH - video): 0
ACK_START rows: 1
ACK_STOP rows: 1
ACK_STOP count field: 242

First CAMERA_HIGH row:
{'eventType': 'CAMERA_HIGH', 'unixTime': '1147214194413594', 'rp2040Time': '343388647186', 'side': 'nan', 'count': '1', 'duration': '69420', 'latency': '69420', 'value': '69420', 'context': 'Pellet_Available', 'reason': 'nan'}

Last CAMERA_HIGH row:
{'eventType': 'CAMERA_HIGH', 'unixTime': '1147214202447274', 'rp2040Time': '343396680865', 'side': 'nan', 'count': '242', 'duration': '69420', 'latency': '69420', 'value': '69420', 'context': 'Pellet_Available', 'reason': 'nan'}

Last ACK_STOP row:
{'eventType': 'ACK_STOP', 'unixTime': '1147303396683939', 'rp2040Time': '343396687048', 'side': 'nan', 'count': '242', 'duration': '69420', 'latency': '69420', 'value': '69420', 'context': 'Pellet_Available', 'reason': 'nan'}